In [1]:
import pandas as pd 
import numpy as np 

In [2]:
def get_adj_risk_score():
    results = pd.read_csv("recidivism_results/pair_results_size100.csv")
    results['response'] = np.where((results['response'] == "Person 1") | (results['response'] == "Person 2"), results['response'], "Removed response")

    rating_counts_df = pd.concat([
    results.loc[results['response'] == 'Person 1', 'id.x'],
    results.loc[results['response'] == 'Person 2', 'id.y']
    ])

    rate_df = rating_counts_df.value_counts().reset_index()
    rate_df.columns = ['id', 'risk_score']

    unknown_counts_df = pd.concat([
    results.loc[results['response'] == 'Removed response', 'id.x'],
    results.loc[results['response'] == 'Removed response', 'id.y']
    ])

    unknown_df = unknown_counts_df.value_counts().reset_index()
    unknown_df.columns = ['id', 'unknown_count']


    judges = pd.read_csv("preprocessed_data/judges_pairs_preprocessed_unpaired_size100.csv")
    results_df = pd.merge(pd.merge(judges, rate_df, on = 'id', how = 'left'), unknown_df, on = 'id', how = 'left')
    results_df['risk_score'] = results_df['risk_score'].fillna(0)
    results_df['unknown_count'] = results_df['unknown_count'].fillna(0)
    results_df['risk_score_adj'] = results_df['risk_score']/(results_df['pair_group_size'] - results_df['unknown_count'])

    return results_df

In [3]:
judges = pd.read_csv("judges.csv")
judges['id'] = range(1, len(judges) + 1)
judges = judges[['id', 'calendar1', 'calendar2', 'calendar3', 'calendar4', 'calendar5', 'calendar6', 'calendar7', 'calendar8']]

judges_results = get_adj_risk_score() 
judges_results = pd.merge(judges_results, judges, on = 'id', how = 'left')

In [7]:
import statsmodels.api as sm

# no pair score
judges_results['gender_mod'] = np.where(judges_results['gender'] =="male", 0, 1)
judges_results['nonblack_mod'] = np.where(judges_results['nonblack'] == "not Black", 0, 1)
y = judges_results[['laterarr']]
X = judges_results[['oob_preds', 'age', 'gender_mod', 'nonblack_mod', 'marijuana', 'cocaine', 'crack', 'heroin', 'pcp', 'otherdrug', 'nondrug', 'priorarr', 'priorfelarr', 'priordrugarr', 'priorfeldrugarr', 'priorcon', 'priorfelcon', 'priordrugcon', 'priorfeldrugcon', 'pwid', 'dist', 'calendar1', 'calendar2', 'calendar3', 'calendar4', 'calendar5', 'calendar6', 'calendar7', 'calendar8']]
X = sm.add_constant(X)

model1 = sm.Logit(y, X).fit()

print(model1.summary())

Optimization terminated successfully.
         Current function value: 0.641158
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:               laterarr   No. Observations:                 1003
Model:                          Logit   Df Residuals:                      973
Method:                           MLE   Df Model:                           29
Date:                Sat, 28 Mar 2026   Pseudo R-squ.:                 0.07314
Time:                        13:23:26   Log-Likelihood:                -643.08
converged:                       True   LL-Null:                       -693.83
Covariance Type:            nonrobust   LLR p-value:                 5.644e-10
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               1.5514      0.493      3.145      0.002       0.585       2.518
oob_preds     

In [8]:
X2 = judges_results[['oob_preds', 'risk_score_adj', 'age', 'gender_mod', 'nonblack_mod', 'marijuana', 'cocaine', 'crack', 'heroin', 'pcp', 'otherdrug', 'nondrug', 'priorarr', 'priorfelarr', 'priordrugarr', 'priorfeldrugarr', 'priorcon', 'priorfelcon', 'priordrugcon', 'priorfeldrugcon', 'pwid', 'dist', 'calendar1', 'calendar2', 'calendar3', 'calendar4', 'calendar5', 'calendar6', 'calendar7', 'calendar8']]
X2= sm.add_constant(X2)

model2 = sm.Logit(y, X2).fit()

print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.641136
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:               laterarr   No. Observations:                 1003
Model:                          Logit   Df Residuals:                      972
Method:                           MLE   Df Model:                           30
Date:                Sat, 28 Mar 2026   Pseudo R-squ.:                 0.07317
Time:                        13:23:42   Log-Likelihood:                -643.06
converged:                       True   LL-Null:                       -693.83
Covariance Type:            nonrobust   LLR p-value:                 1.062e-09
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               1.5534      0.493      3.148      0.002       0.586       2.520
oob_preds     